In [ ]:
%pip -q install -U deepl datasets evaluate sacrebleu

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import re
import random
import xml.etree.ElementTree as ET
from typing import Optional, Dict, Tuple, List

import numpy as np
import deepl
import evaluate
from datasets import Dataset

In [ ]:
TMX_ROOT = r"J:\FINAL PROJECT"
TMX_EN_ES = os.path.join(TMX_ROOT, "data", "en-es.tmx")
TMX_EN_PT = os.path.join(TMX_ROOT, "data", "en-pt.tmx")

PROJECT_DIR = "./deepl_scielo_better"
os.makedirs(PROJECT_DIR, exist_ok=True)

SEED = 42

PAIR = "en-es"   

TRAIN_N = 800
VALID_N = 100
TEST_N = 100

BATCH_SIZE = 20
FORMALITY = "prefer_more"   

random.seed(SEED)
np.random.seed(SEED)

print("Project dir:", PROJECT_DIR)

Project dir: ./deepl_scielo_better


In [ ]:

os.environ["DEEPL_API_KEY"] = "56f67712-73e4-4e7f-841c-b922ffcb30ed:fx"


DEEPL_API_KEY = os.getenv("DEEPL_API_KEY")
assert DEEPL_API_KEY, "DEEPL_API_KEY is not set"

In [ ]:
translator = deepl.DeepLClient(DEEPL_API_KEY)
print("DeepL client initialized.")

DeepL client initialized.


In [ ]:
_LANG_ALIASES = {
    "en": {"en", "eng", "en-us", "en-gb"},
    "es": {"es", "spa", "es-es", "es-la", "es-mx", "es-419"},
    "pt": {"pt", "por", "pt-pt", "pt-br"},
}

def _norm_lang(code: str) -> str:
    c = (code or "").lower()
    for k, aliases in _LANG_ALIASES.items():
        if c in aliases:
            return k
    m = re.match(r"^([a-z]{2})[-_][a-z]{2,3}$", c)
    if m and m.group(1) in _LANG_ALIASES:
        return m.group(1)
    return c

In [ ]:
def _extract_pair_from_tu(tu_elem, src_key, tgt_key):
    texts = {}
    for tuv in tu_elem.findall(".//tuv"):
        lang = _norm_lang(
            tuv.attrib.get("{http://www.w3.org/XML/1998/namespace}lang")
            or tuv.attrib.get("lang") or ""
        )
        seg = tuv.find(".//seg")
        if seg is not None and seg.text is not None:
            texts[lang] = seg.text.strip()

    if src_key in texts and tgt_key in texts:
        return texts[src_key], texts[tgt_key]
    return None


def read_tmx_pairs(path: str, src_key: str, tgt_key: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"TMX not found: {path}")

    src, tgt = [], []

    for event, elem in ET.iterparse(path, events=("end",)):
        if elem.tag.lower().endswith("tu"):
            pair = _extract_pair_from_tu(elem, src_key, tgt_key)
            if pair:
                s, t = pair
                if s and t:
                    src.append(s)
                    tgt.append(t)
            elem.clear()

    return src, tgt

In [ ]:
TAG = re.compile(r"</?[^>]+>")
WS = re.compile(r"\s+")

def _norm_text(x: str) -> str:
    x = x or ""
    x = TAG.sub(" ", x)
    x = WS.sub(" ", x)
    return x.strip()


def global_clean_and_dedup(src_all, tgt_all, min_ratio=0.5, max_ratio=1.8, min_len=5):
    rows = []

    for s, t in zip(src_all, tgt_all):
        s2, t2 = _norm_text(s), _norm_text(t)

        if not s2 or not t2:
            continue

        sw, tw = s2.split(), t2.split()

        if len(sw) < min_len or len(tw) < min_len:
            continue

        ratio = len(tw) / max(1, len(sw))
        if ratio < min_ratio or ratio > max_ratio:
            continue

        rows.append((s2, t2))

    seen = set()
    uniq = []

    for s2, t2 in rows:
        if s2 in seen:
            continue
        seen.add(s2)
        uniq.append((s2, t2))

    return [s for s, _ in uniq], [t for _, t in uniq]

In [ ]:
def split_train_valid_test(src_clean, tgt_clean, train_n=800, valid_n=100, test_n=100, seed=42):
    total_needed = train_n + valid_n + test_n
    assert len(src_clean) >= total_needed, f"Need at least {total_needed} cleaned pairs, got {len(src_clean)}"

    idxs = list(range(len(src_clean)))
    rng = random.Random(seed)
    rng.shuffle(idxs)

    train_idx = idxs[:train_n]
    valid_idx = idxs[train_n:train_n + valid_n]
    test_idx = idxs[train_n + valid_n:train_n + valid_n + test_n]

    def build(idxs):
        return Dataset.from_dict({
            "src_text": [src_clean[i] for i in idxs],
            "tgt_text": [tgt_clean[i] for i in idxs],
        })

    return build(train_idx), build(valid_idx), build(test_idx)

In [ ]:
if PAIR == "en-es":
    print("Reading EN-ES TMX:", os.path.abspath(TMX_EN_ES))
    src_all, tgt_all = read_tmx_pairs(TMX_EN_ES, "en", "es")
    LANG_NAME = "Spanish"
elif PAIR == "en-pt":
    print("Reading EN-PT TMX:", os.path.abspath(TMX_EN_PT))
    src_all, tgt_all = read_tmx_pairs(TMX_EN_PT, "en", "pt")
    LANG_NAME = "Portuguese"
else:
    raise ValueError("PAIR must be 'en-es' or 'en-pt'")

print("Raw pairs:", len(src_all))

Reading EN-PT TMX: J:\FINAL PROJECT\data\en-pt.tmx
Raw pairs: 2747083


In [ ]:
src_clean, tgt_clean = global_clean_and_dedup(src_all, tgt_all)

print("Cleaned pairs:", len(src_clean))

train_ds, valid_ds, test_ds = split_train_valid_test(
    src_clean, tgt_clean,
    train_n=TRAIN_N,
    valid_n=VALID_N,
    test_n=TEST_N,
    seed=SEED
)

print("Train:", len(train_ds))
print("Valid:", len(valid_ds))
print("Test :", len(test_ds))

Cleaned pairs: 2679175
Train: 800
Valid: 100
Test : 100


In [ ]:
EQ_PATTERN   = re.compile(r"\b(?:log|ln|exp|sin|cos|tan)\s*\([^)]*\)")
LATEX_EQ     = re.compile(r"\$[^$]+\$")
CIT_PATTERN  = re.compile(r"[A-Z][a-z]+ et al\.\s*\(\d{4}\)")
PERC_PATTERN = re.compile(r"\d+(?:[.,]\d+)?\s*%")
NUM_PATTERN  = re.compile(r"\d+(?:[.,]\d+)?")

PROTECT_PATTERNS = [LATEX_EQ, EQ_PATTERN, CIT_PATTERN, PERC_PATTERN, NUM_PATTERN]


def rbmt_protect(text: str) -> Tuple[str, Dict[str, str]]:
    spans = []

    for pat in PROTECT_PATTERNS:
        for m in pat.finditer(text):
            spans.append((m.start(), m.end(), m.group(0)))

    spans.sort(key=lambda x: (x[0], -(x[1] - x[0])))

    non_overlapping = []
    last_end = -1

    for s, e, v in spans:
        if s >= last_end:
            non_overlapping.append((s, e, v))
            last_end = e

    out = []
    cur = 0
    mp = {}
    k = 0

    for s, e, v in non_overlapping:
        if cur < s:
            out.append(text[cur:s])

        ph = f"__PH_{k}__"
        mp[ph] = v
        out.append(ph)

        cur = e
        k += 1

    if cur < len(text):
        out.append(text[cur:])

    return "".join(out), mp


def rbmt_restore(text: str, mp: Dict[str, str]) -> str:
    for ph, val in mp.items():
        text = text.replace(ph, val)
    return text

In [ ]:
if PAIR == "en-es":
    MANUAL_GLOSSARY = {
        "systematic review": "revisión sistemática",
        "confidence interval": "intervalo de confianza",
        "primary health care": "atención primaria de salud",
        "social vulnerability": "vulnerabilidad social",
        "psychosocial risk": "riesgo psicosocial",
        "health care": "atención de salud",
    }
else:
    MANUAL_GLOSSARY = {
        "systematic review": "revisão sistemática",
        "confidence interval": "intervalo de confiança",
        "primary health care": "atenção primária à saúde",
        "social vulnerability": "vulnerabilidade social",
        "psychosocial risk": "risco psicossocial",
        "health care": "atenção à saúde",
    }

print("Glossary entries:", len(MANUAL_GLOSSARY))

Glossary entries: 6


In [ ]:
def build_glossary_entries(manual_dict: Dict[str, str]):
    entries = []
    seen = set()

    for s, t in manual_dict.items():
        key = s.strip().lower()
        if key in seen:
            continue
        seen.add(key)
        entries.append((s, t))

    return entries


def maybe_create_glossary(target_lang: str):
    entries = build_glossary_entries(MANUAL_GLOSSARY)

    if not entries:
        return None

    name = f"scielo_{PAIR}_{target_lang.lower()}_{SEED}"

    try:
        glossary = translator.create_multilingual_glossary(
            name=name,
            dictionaries=[
                deepl.MultilingualGlossaryDictionaryEntries(
                    source_lang="EN",
                    target_lang=target_lang,
                    entries=entries
                )
            ]
        )
        print("Created glossary:", glossary.glossary_id)
        return glossary.glossary_id
    except Exception as e:
        print("Glossary creation failed, continuing without glossary.")
        print("Reason:", e)
        return None

In [ ]:
if PAIR == "en-pt":
    CANDIDATE_TARGETS = ["PT-PT", "PT-BR"]
elif PAIR == "en-es":
    CANDIDATE_TARGETS = ["ES", "ES-419"]
else:
    CANDIDATE_TARGETS = []

print("Candidates:", CANDIDATE_TARGETS)

Candidates: ['PT-PT', 'PT-BR']


In [ ]:
def translate_batch(texts: List[str], target_lang: str, glossary_id=None):
    protected_texts = []
    maps = []

    for text in texts:
        p, mp = rbmt_protect(text)
        protected_texts.append(p)
        maps.append(mp)

    kwargs = dict(
        source_lang="EN",
        target_lang=target_lang,
        formality=FORMALITY,
        preserve_formatting=False,
    )

    # Add glossary only if available
    if glossary_id is not None:
        kwargs["glossary_id"] = glossary_id

    results = translator.translate_text(protected_texts, **kwargs)

    if not isinstance(results, list):
        results = [results]

    outs = []
    for result, mp in zip(results, maps):
        outs.append(rbmt_restore(result.text, mp))

    return outs

In [ ]:
def predict_dataset(ds: Dataset, target_lang: str, glossary_id=None, batch_size=20):
    preds = []
    refs = []

    for start in range(0, len(ds), batch_size):
        batch_src = ds[start:start + batch_size]["src_text"]
        batch_ref = ds[start:start + batch_size]["tgt_text"]

        batch_hyp = translate_batch(
            batch_src,
            target_lang=target_lang,
            glossary_id=glossary_id
        )

        preds.extend(batch_hyp)
        refs.extend(batch_ref)

        print(f"Translated {min(start + batch_size, len(ds))}/{len(ds)}")

    return preds, refs

In [ ]:
bleu_metric = evaluate.load("sacrebleu")
ter_metric = evaluate.load("ter")

def evaluate_preds(preds: List[str], refs: List[str]):
    refs_ll = [[r] for r in refs]

    bleu = bleu_metric.compute(
        predictions=preds,
        references=refs_ll,
        use_effective_order=True
    )["score"]

    ter = ter_metric.compute(
        predictions=preds,
        references=refs_ll
    )["score"]

    exact = float(np.mean([p == r for p, r in zip(preds, refs)]) * 100.0)

    return {
        "bleu": float(bleu),
        "ter": float(ter),
        "exact_match_%": exact
    }

In [ ]:
valid_results = []

for target_lang in CANDIDATE_TARGETS:
    print("\n" + "=" * 80)
    print("Testing target:", target_lang)

    glossary_id = maybe_create_glossary(target_lang)

    preds, refs = predict_dataset(
        valid_ds,
        target_lang=target_lang,
        glossary_id=glossary_id,
        batch_size=BATCH_SIZE
    )

    scores = evaluate_preds(preds, refs)

    row = {
        "target_lang": target_lang,
        "glossary_id": glossary_id,
        "bleu": scores["bleu"],
        "ter": scores["ter"],
        "exact_match_%": scores["exact_match_%"],
    }
    valid_results.append(row)

    print(f"VALID {target_lang} -> BLEU {scores['bleu']:.2f} | TER {scores['ter']:.2f} | Exact {scores['exact_match_%']:.2f}%")


Testing target: PT-PT
Glossary creation failed, continuing without glossary.
Reason: module 'deepl' has no attribute 'MultilingualGlossaryDictionaryEntries'
Translated 20/100
Translated 40/100
Translated 60/100
Translated 80/100
Translated 100/100
VALID PT-PT -> BLEU 40.25 | TER 45.67 | Exact 0.00%

Testing target: PT-BR
Glossary creation failed, continuing without glossary.
Reason: module 'deepl' has no attribute 'MultilingualGlossaryDictionaryEntries'
Translated 20/100
Translated 40/100
Translated 60/100
Translated 80/100
Translated 100/100
VALID PT-BR -> BLEU 44.10 | TER 42.73 | Exact 0.00%


In [ ]:
best_row = max(valid_results, key=lambda x: x["bleu"])

BEST_TARGET = best_row["target_lang"]
BEST_GLOSSARY_ID = best_row["glossary_id"]

print("Best target:", BEST_TARGET)
print("Best glossary:", BEST_GLOSSARY_ID)
print("Best VALID BLEU:", round(best_row["bleu"], 2))

Best target: PT-BR
Best glossary: None
Best VALID BLEU: 44.1


In [ ]:
test_preds, test_refs = predict_dataset(
    test_ds,
    target_lang=BEST_TARGET,
    glossary_id=BEST_GLOSSARY_ID,
    batch_size=BATCH_SIZE
)

test_scores = evaluate_preds(test_preds, test_refs)

print(f"\nTEST {PAIR} DeepL ({BEST_TARGET}) -> BLEU {test_scores['bleu']:.2f} | TER {test_scores['ter']:.2f} | Exact {test_scores['exact_match_%']:.2f}%")

Translated 20/100
Translated 40/100
Translated 60/100
Translated 80/100
Translated 100/100

TEST en-pt DeepL (PT-BR) -> BLEU 45.36 | TER 42.95 | Exact 0.00%


In [ ]:
for i in range(min(10, len(test_preds))):
    print("=" * 100)
    print("SRC :", test_ds[i]["src_text"])
    print("REF :", test_refs[i])
    print("HYP :", test_preds[i])

SRC : Furthermore, other authors affirm that bullying is an independent variable associated with an increase in physical violence among peers and that there is a general association between bullying and truancy and/or skipping classes.
REF : Além disso, outros autores afirmam que obullying é variável independente para o aumento da violência física entre pares e, que de uma maneira geral, está associado com não ir à escola ou cabular aula.
HYP : Além disso, outros autores afirmam que o bullying é uma variável independente associada a um aumento da violência física entre os colegas e que há uma associação geral entre o bullying e a evasão escolar e/ou a falta às aulas.
SRC : After the release of excitatory amino acids, peptides, and neurotrophins and their binding to specific receptors, there is activation of second messengers, such as cAMP, PKA, PKC, phosphatidylinositol, phospholipase C, and phospholipase A2.
REF : Após a liberação de aminoácidos excitatórios, peptídeos e neurotrofinas

In [ ]:
preds_no_gloss, refs_no_gloss = predict_dataset(
    valid_ds,
    target_lang=BEST_TARGET,
    glossary_id=None,
    batch_size=BATCH_SIZE
)

scores_no_gloss = evaluate_preds(preds_no_gloss, refs_no_gloss)
print(f"WITHOUT GLOSSARY -> BLEU {scores_no_gloss['bleu']:.2f} | TER {scores_no_gloss['ter']:.2f}")

Translated 20/100
Translated 40/100
Translated 60/100
Translated 80/100
Translated 100/100
WITHOUT GLOSSARY -> BLEU 43.83 | TER 42.99
